# 1 · bdopt overview

**What you will do.** Read an EnergyPlus model, say which of its values you want to vary, say what you want to measure, simulate a few designs and look at how the inputs relate to the results.

**You need.** Nothing else: this is the first notebook. `test.idf`, the weather file and `window_to_wall.py` are in this folder.

**Run time.** about 4 minutes (it runs 12 one-year EnergyPlus simulations, two at a time).

The four notebooks of this series use the same building model (`test.idf`, a small house) and the same weather file
(Turin). Work in order: 1 *bdopt overview* → 2 *surrogate modelling* → 3 *optimization* → 4 *adaptive sampling*. Cells
marked **Your turn** are exercises: change something and observe what happens.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
plt.rcParams.update({'font.size': 12})

# bdopt: a toolbox for surrogate-based design optimisation of buildings

bdopt integrates building simulation and surrogate modelling for design exploration and optimisation.

It enables:

+ EnergyPlus model reading and editing: the IDF is read as text, a design variable says which field of which objects it writes
+ Evaluation of a parameter space and of objectives in a specific weather location (`EnergyPlusEvaluator`)
+ Generation of samples with `bdopt.sampling` (Latin hypercube, Sobol, Halton, full factorial, ...)
+ Surrogate models with `bdopt.surrogate`: Gaussian processes, neural networks, tree ensembles, ... all built on [scikit-learn](https://scikit-learn.org/stable/)
+ Adaptive sampling (`bdopt.adaptive`), model exploration and multi-objective optimisation (`bdopt.optimize`, NSGA-II)
+ Plots of all the above (`bdopt.viz`)

![Image](image/flow_diagram.PNG)

*Figure adapted from the BESOS documentation (Westermann et al. 2021, GPL-3.0): the names of the toolbox were changed.*

### Words used in this course

| Term | Meaning in bdopt |
|---|---|
| **Selector** | says *where* in the building model a value is written (a field of one or more objects, or a function that edits the model) |
| **Descriptor** | says *which values* are allowed: `RangeParameter` (an interval), `IntegerParameter` (whole numbers), `CategoryParameter` (a list) |
| **Parameter** (design variable) | a selector plus a descriptor, with a name: one column of the table of designs |
| **Objective** | a quantity read from the simulation results, for example an energy meter |
| **Problem** | parameters + objectives |
| **Evaluator** | turns a table of designs into a table of objectives: by simulating (`EnergyPlusEvaluator`) or with a Python function (`FunctionEvaluator`) |
| **Sample** | one design (one row); *sampling* means choosing many designs to evaluate |
| **Surrogate** | a fast statistical model trained on simulated samples, used in place of the simulator |
| **Pareto front** | the designs that no other design beats on every objective at once |

## What this notebook does
It performs a parametric analysis of a building design with EnergyPlus and bdopt. We load the model `test.idf`, define the parameters to vary and the objectives to measure, run the model for a single design and then for a set of samples, and plot the results.

## Read and explore the EnergyPlus model

The building is defined by the Information Data File (`.idf`), a plain text file. bdopt reads IDF files; a model in the newer epJSON format has to be converted first (EnergyPlus ships the `ConvertInputFormat` program for that).

`load_idf` reads the file into a Python object (`IDF`): a list of objects, each with its class name and its field values as text. It does not need EnergyPlus to be installed.

In [ ]:
from bdopt import EnergyPlus, load_idf

In [ ]:
idf = load_idf('test.idf')

In [ ]:
idf

bdopt does not draw the geometry. To see the building: `test.idf` already contains an `Output:Surfaces:Drawing` object (report type `DXF`), so every EnergyPlus run writes a DXF drawing of the geometry in its run folder (`evaluator.evaluate(..., keep_dirs=True)` keeps the run folders); any CAD viewer opens it. To ask for it in another model: `idf.add('Output:Surfaces:Drawing', 'DXF')`.

In [ ]:
# the classes of the model and how many objects each one has
{c: len(idf.by_class(c)) for c in sorted(idf.classes())}

Let's check what `MATERIAL` are in the model. The IDF holds only values; the field names come from the Input Data Dictionary (IDD) of the EnergyPlus installation:

In [ ]:
idd = EnergyPlus.find().idd                 # field names of every EnergyPlus class

def fields_of(obj):
    """An IDF object as {field name: value}."""
    return dict(zip(idd.fields(obj.class_name), obj.fields))

idf.by_class('Material')[0]

In [ ]:
fields_of(idf.by_class('Material')[0])

In [ ]:
for material in idf.by_class('Material'):
    f = fields_of(material)
    print("{} - {}[m]".format(f['Name'], f['Thickness']))

In [ ]:
for window in idf.by_class('FenestrationSurface:Detailed'):
    print(fields_of(window))

In [ ]:
for construction in idf.by_class('Construction'):
    print(fields_of(construction))

# Evaluators

An *evaluator* turns a table of designs (one column per parameter) into a table of results (one column per objective). `bdopt` provides:
- the EnergyPlus evaluator (`EnergyPlusEvaluator`): simulates every design
- the function evaluator (`FunctionEvaluator`): computes the objectives with a Python function (a formula, a surrogate model, another simulator)

## FunctionEvaluator
A `FunctionEvaluator` takes a function with the right number of inputs and gives back one value per objective.

A `Problem` is made of parameters and objectives. Here no building is edited, so the parameter does not need a selector that writes into a model: `NullSelector` does nothing. The objectives are only names.

A `Problem` object represents inputs and outputs.

In [ ]:
from bdopt import FunctionEvaluator, NullSelector, Parameter, Problem, RangeParameter

problem = Problem([Parameter(NullSelector('x'), RangeParameter(0, 10), name='x')],
                  ['f1', 'f2'])    # a problem which takes 1 input and produces 2 outputs

# These names are used on the headings of the tables generated by the problem and by the evaluators that use it
print(problem.names())

In [ ]:
def function(values):  # define a dummy function f1(x)=x, f2(x)=x^2
    return (values['x'], values['x'] ** 2)

def function2(values):  # define a dummy function f1(x)=x, f2(x)=x*2
    return (values['x'], values['x'] * 2)

print("Functions values at 4 are:")

evaluator_1 = FunctionEvaluator(problem, function)
evaluator_2 = FunctionEvaluator(problem, function2)
print(evaluator_1([4]))
print(evaluator_2([4]))

## EnergyPlusEvaluator

![Image](image/setting_up_the_evaluator.PNG)

Defines how we will evaluate the building;
- what external weather conditions is the building experiencing,
- what properties of the building will we be changing, and
- what are some of the performance metrics of the building that we want to explore.

The weather conditions are specified in the EnergyPlus Weather file (EPW). The properties we will change in the building will be defined in the parameter space. In the objectives we will specify the output performance metrics we wish to extract so that we can explore them later.

*Figure from the BESOS documentation (Westermann et al. 2021, GPL-3.0).*

In [ ]:
from bdopt import EnergyPlusEvaluator, FieldSelector

### What have we imported?
`EnergyPlusEvaluator` manages EnergyPlus simulations for a single problem. It needs a `Problem` (the `Parameters` to modify and the `objectives` to report), a building model and a weather file.
A problem (`problem = parameters + objectives`) can be applied to any building model (`evaluator = problem + building + weather`).

Parameters, like `Parameter`, describe a single variable for the building.
They are composed of a selector and a value descriptor.

## Parameters and Objectives
A problem `Parameter` is composed of one *selector* that identifies which part of the building model to modify, and a *descriptor* that specifies what kinds of values are valid for the parameter.

In [ ]:
for materials in idf.by_class('Material'):
    print(materials)

In [ ]:
object_name = 'XPS Extruded Polystyrene  - CO2 Blowing_0041'
for materials in idf.by_class('Material'):
    if materials.name == object_name:
        print(fields_of(materials))

## Field Selectors
A `FieldSelector` modifies one field of the objects of a class in a building model.

The example building loaded here contains a `Material` class object named `XPS Extruded Polystyrene  - CO2 Blowing_0041` which has a `Thickness` field.
We make a selector that modifies this insulation thickness.

In [ ]:
insulation = FieldSelector(class_name='Material', object_name='XPS Extruded Polystyrene  - CO2 Blowing_0041', field_name='Thickness')

In [ ]:
insulation

### Setting all fields on objects of a certain class
If you have multiple objects of the same type that all share the same field to modify, you can modify them all with one selector by setting `object_name` to `'*'`. A list of names (`object_name=['Zone A', 'Zone B']`) selects exactly those objects, and `where={'Surface Type': 'Window'}` keeps only the objects whose field has that value.

In [ ]:
[fields_of(l)['Name'] for l in idf.by_class('Lights')]       # the names of the Lights objects of the model

Our building has several Lights, and all of the lights have a field called `Watts per Zone Floor Area`

In [ ]:
lights_selector = FieldSelector(class_name='Lights', object_name='*', field_name='Watts per Zone Floor Area')

In [ ]:
lights_selector

In [ ]:
# this selector affects the following objects
lights_selector.targets(idf, idd)

A selector can write a value into a model: `apply` sets all the fields at once. We do it on a *copy* of the model, so that `idf` stays as it was. Reading the field back is reading the object.

In [ ]:
index = idd.field_index('Lights', 'Watts per Zone Floor Area')     # position of the field in the object
edited = idf.copy()
lights_selector.apply(edited, 11, idd)
[l.get(index) for l in lights_selector.targets(edited, idd)]

### Running a single value and single parameter problem

A `Parameter` object is created by giving it the selector and the values the variable may take.

In [ ]:
parameters = [Parameter(selector=insulation, value_descriptor=RangeParameter(0.01, 1.0), name='Insulation Thickness')]
parameters

Now we have to define the `objectives` and the problem

In [ ]:
objectives = ['Electricity:Facility', 'DistrictHeating:Facility', 'DistrictCooling:Facility']   # EnergyPlus meters, summed over the run period; they are made into `MeterReader` objects
problem = Problem(parameters, objectives)    # problem = parameters + objectives
problem

In [ ]:
epw_file = 'ITA_Torino.160590_IWEC.epw'
evaluator = EnergyPlusEvaluator(problem, idf, epw_file,
                                out_dir='/tmp/bdopt_runs',   # work files go to the container's own disk (the shared folder is much slower)
                                err_dir='outputdir',         # runs that fail are kept here, with the EnergyPlus error file
                                processes=2)                 # evaluator = problem + building + weather
print(evaluator.eplus)

Now we run the evaluator with the insulation thickness set to 0.5 m. A list is read in the order of the parameters of the problem.

In [ ]:
result = evaluator([0.5])
values = dict(zip(objectives, result))

In [ ]:
result1 = evaluator([1])
values1 = dict(zip(objectives, result1))

In [ ]:
for key, value in values.items():
    print(key, " :: ", "{0:.2f}".format(value / 3.6e6), "kWh")

print()
print()

for key, value1 in values1.items():
    print(key, " :: ", "{0:.2f}".format(value1 / 3.6e6), "kWh")

**Your turn.** Evaluate the model with an insulation thickness of 0.05 m and of 1.0 m and compare the three objectives.
Which one changes most? Does more insulation always reduce the total energy? Think about what insulation does in summer.

In [ ]:
# your code here: evaluator([...])

(Objectives are in joules, the unit EnergyPlus reports. `MeterReader('Electricity:Facility', unit='kWh')` gives them in kWh directly.)

# Giving more than one value to a Parameter with Descriptors


There are three value descriptors: `RangeParameter`, `IntegerParameter` and `CategoryParameter`.


`RangeParameter` is a descriptor that indicates a parameter that can take on values from an interval.
`IntegerParameter` takes whole numbers from an interval, both ends included.
`CategoryParameter` is a descriptor that indicates a parameter that can take on values from a list.

In [ ]:
from bdopt import CategoryParameter, IntegerParameter

### RangeParameters
$min \leq x \leq max$

In [ ]:
zero_to_one_exclusive = RangeParameter(min_val=0.01, max_val=0.99)

### CategoryParameters
A list of options (text or numbers).

In [ ]:
text_example = CategoryParameter(options=['a', 'b', 'c', 'other'])
single_digit_integers = IntegerParameter(8, 12)

In [ ]:
zero_to_one_exclusive

We can combine this with the `Selector` above to get a `Parameter`:

In [ ]:
insulation = FieldSelector(class_name='Material', object_name='XPS Extruded Polystyrene  - CO2 Blowing_0041', field_name='Thickness')

insulation_param = Parameter(selector=insulation,
                             value_descriptor=zero_to_one_exclusive,
                             name='Insulation Thickness')
print(insulation_param)
parameters = [insulation_param]

In [ ]:
lights_param = Parameter(
    lights_selector,
    value_descriptor=single_digit_integers,
    name="Lights Watts/Area",
)

In [ ]:
lights_param

### Several fields of an object, several objects

bdopt has no short-cut that builds many parameters from a dictionary: you write one `Parameter` per variable, with the class of the object. This also avoids the ambiguity of an object name that exists in two classes.

The glazing material `3` of this model has a `Conductivity` and a `Solar Transmittance at Normal Incidence`:

In [ ]:
[fields_of(m)['Name'] for m in idf.by_class('WindowMaterial:Glazing')]

In [ ]:
more_parameters = [
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Conductivity'),
              RangeParameter(0.1, 1), name='Glazing conductivity'),
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Solar Transmittance at Normal Incidence'),
              RangeParameter(0.01, 0.8), name='Glazing solar transmittance'),
]

In [ ]:
for p in more_parameters:
    print(p)

Remember: a selector with `'*'`, with a list of object names or with `where=` writes the *same* value into every object it matches, and it is still **one** column of the design table. When the places differ in class or field (for example one set point written into several schedules and into a thermostat), join several selectors in a `MultiSelector`: again one variable, one column.

### Parameter scripts using a Generic Selector

A parameter can also be defined by a function that takes the model and a value and edits the model. These functions are specific to the format of a model and can perform any transformation. In bdopt this is a `GenericSelector(setter)`: `setter(idf, value)` is called, on a copy of the model, with the value of the parameter.

A window-to-wall ratio changes the geometry, not a single field, so it is written as a `GenericSelector`. The course provides it in a small file next to this notebook, `window_to_wall.py`: it scales the windows of every outdoor wall about their own centre until the windows of that wall add up to the requested fraction of the wall. It creates and deletes nothing. It is meant for models whose windows can grow in place: on `test.idf` every ratio up to 0.83 fits. For a ratio that the windows cannot reach without leaving their wall or overlapping another window or a door, the setter raises a `ValueError` instead of writing a wrong model. The limits are at the top of the file (cell below).

Two things to keep in mind. The ratio is imposed wall by wall on the walls that already have windows, so the ratio of the whole building is lower when some wall has none. And EnergyPlus' own report of the window-to-wall ratio counts the window frame: asking 0.5 on `test.idf` gives 52.4 % in its `EnvelopeSummary` table. It is a tool for the models of this course, not a general one.

In [ ]:
print(open('window_to_wall.py').read())

Here we define the window-to-wall ratio of all the walls of the model to be between 10% and 80%.

In [ ]:
from window_to_wall import wwr

In [ ]:
window_to_wall = wwr(0.1, 0.8)        # a ready-made Parameter(GenericSelector(...), RangeParameter(0.1, 0.8))
window_to_wall

In [ ]:
parameters = [insulation_param] + more_parameters + [window_to_wall] + [lights_param]

In [ ]:
parameters

In [ ]:
problem = Problem(parameters, objectives)

## Sampling
Once we have created `Parameters` with selectors and descriptors, we can generate samples with `bdopt.sampling`. Each function takes the problem and returns a table with one column per parameter.

In [ ]:
from bdopt import sampling

In [ ]:
samples = sampling.factorial(problem, levels=3)     # full factorial: every combination of 3 levels per variable (the integer parameter has 5 values: capped at 3)
print(len(samples), 'designs')
samples.head()

In [ ]:
samples = sampling.lhs(problem, 10, seed=0)         # Latin hypercube
samples

### Which designs to simulate?

A one-year simulation of this small house takes about 14 seconds, a large building minutes, and a study needs hundreds of them: we want few designs that still *cover* the design space.
- **Full factorial** tries every combination of a few levels per variable: simple, but the number of designs grows as `levels ** variables`.
- **Latin hypercube (`lhs`)** divides the range of every variable in `n` slices and uses each slice exactly once: `n` designs for any number of variables.
- **Sobol** and **Halton** are low-discrepancy sequences: they fill the space evenly and can be extended with more points.

Below, 16 designs chosen by each method, seen on two of the variables. Random points (grey) leave gaps and clusters.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True, sharey=True)
x_name, y_name = problem.parameter_names[0], problem.parameter_names[3]
x_range, y_range = problem.parameters[0].value_descriptor, problem.parameters[3].value_descriptor
rng = np.random.default_rng(0)
axes[0].scatter(rng.uniform(x_range.min, x_range.max, 16), rng.uniform(y_range.min, y_range.max, 16), color='grey')
axes[0].set_title('random')
for ax, (label, function) in zip(axes[1:], {'lhs': sampling.lhs, 'sobol': sampling.sobol, 'halton': sampling.halton}.items()):
    s = function(problem, 16, seed=0)
    ax.scatter(s[x_name], s[y_name])
    ax.set_title(label)
for ax in axes:
    ax.set_xlabel(x_name)
axes[0].set_ylabel(y_name)
plt.show()

The simulated designs below use `lhs`.

In [ ]:
problem

## Evaluation
Now we can evaluate the samples. We create an EnergyPlus evaluator (`EnergyPlusEvaluator`) with the problem, the IDF describing the building, the weather file and the objectives we want to measure.

In order to keep the outputs of every simulation we can pass `keep_dirs=True` to `evaluate`: one folder per simulation, with all the EnergyPlus output files, is then left in the evaluator's `out_dir`.

In [ ]:
objectives = ['Electricity:Facility', 'DistrictHeating:Facility', 'DistrictCooling:Facility']   # EnergyPlus meters

problem = Problem(parameters, objectives)    # problem = parameters + objectives

In [ ]:
evaluator = EnergyPlusEvaluator(problem, idf, epw_file,
                                out_dir='/tmp/bdopt_runs',
                                err_dir='outputdir',
                                processes=2)
outputs = evaluator.evaluate(samples, keep_input=True)

In [ ]:
outputs

In [ ]:
outputs.describe()

In [ ]:
outputs = outputs.sort_values(by=objectives[1])
ax = outputs.plot.bar(subplots=True, legend=None, figsize=(12, 20))

## Visualising the explored space

A better way to analyse the results is by looking at scatter plots of the inputs versus the outputs.
This enables us to visually see strong relationships of inputs and outputs.

In [ ]:
from bdopt import viz

In [ ]:
plt.rcParams.update({'font.size': 11})
viz.pairplot(outputs, problem, size=2.6)
plt.show()

### Correlation heat map
Another way to analyse the impact of the inputs on the outputs is by analysing the correlation.
A common metric is the Pearson correlation coefficient:

$$ r = \frac{N\sum XY - \sum X \sum Y}{\sqrt{\left[N\sum X^2 - \left(\sum X\right)^2\right]\left[N\sum Y^2 - \left(\sum Y\right)^2\right]}} $$

where $N$ is the number of samples, $X$ is the vector of observations of variable 1 (e.g. the insulation thickness) and $Y$ is
the vector of observations of variable 2 (e.g. the heating demand).
The closer $r$ is to one the stronger the correlation, and similarly for negative one and negative correlation.

To visualize the correlation coefficients of all inputs and outputs, we can plot a heatmap:

In [ ]:
corr = outputs.corr()
corr

In [ ]:
corr = corr.drop(objectives, axis=1)
corr = corr.drop(samples.columns, axis=0)
corr

In [ ]:
viz.correlation_heatmap(outputs, problem)
plt.show()

**Your turn.** Which input has the strongest effect on the heating demand, and on the cooling demand? Does it agree with what you expect from physics?
With only 10 samples, how much do you trust a correlation of 0.3? Repeat the study with `sampling.lhs(problem, 10, seed=1)` and see whether the answer changes.

## Summary
- A **model** is read with `load_idf`; its fields are named by the EnergyPlus IDD.
- A **parameter** = selector (where) + descriptor (which values); a **problem** = parameters + objectives.
- An **evaluator** = problem + building + weather; `EnergyPlusEvaluator` simulates, `FunctionEvaluator` computes.
- **Sampling** chooses the designs to evaluate; plots and correlations show which inputs matter.

Next: **2_surrogate_modelling.ipynb** trains surrogate models on simulated samples.

## Sources and credits

**Origin of this series.** The four notebooks are adapted from the notebooks of the course, which were
written for the toolbox BESOS (Westermann, P., Christiaanse, T.V., Beckett, W., Kovacs, P., Evins, R. (2021). besos: Building and Energy Simulation, Optimization and Surrogate Modelling. *Journal of Open Source Software* 6(60), 2677. https://doi.org/10.21105/joss.02677; code and examples under GPL-3.0, https://gitlab.com/energyincities/besos). They were
rewritten in October 2026 for **bdopt** (MIT licence, https://github.com/LorenzoBottaccioli/bdopt) and extended; the figures marked
above come from those notebooks or from the BESOS documentation.

**Model and weather.** `test.idf` was generated with DesignBuilder for the course. `ITA_Torino.160590_IWEC.epw`: IWEC data,
original source data (c) 2001 American Society of Heating, Refrigerating and Air-Conditioning Engineers (ASHRAE), Inc., Atlanta,
GA, USA, converted to EPW for EnergyPlus; licensed for your own use, not to be passed to third parties without ASHRAE's permission.

**Simulation.** The building is simulated with EnergyPlus version 9.6 (U.S. Department of Energy and partners, BSD-style licence).

**References for this notebook.**
- Crawley, D.B., Lawrie, L.K., Winkelmann, F.C., Buhl, W.F., Huang, Y.J., Pedersen, C.O., Strand, R.K., Liesen, R.J., Fisher, D.E., Witte, M.J., Glazer, J. (2001). EnergyPlus: creating a new-generation building energy simulation program. *Energy and Buildings* 33(4), 319-331. https://doi.org/10.1016/S0378-7788(00)00114-6
- Westermann, P., Christiaanse, T.V., Beckett, W., Kovacs, P., Evins, R. (2021). besos: Building and Energy Simulation, Optimization and Surrogate Modelling. *Journal of Open Source Software* 6(60), 2677. https://doi.org/10.21105/joss.02677
- McKay, M.D., Beckman, R.J., Conover, W.J. (1979). A comparison of three methods for selecting values of input variables in the analysis of output from a computer code. *Technometrics* 21(2), 239-245. https://doi.org/10.1080/00401706.1979.10489755
- Sobol', I.M. (2001). Global sensitivity indices for nonlinear mathematical models and their Monte Carlo estimates. *Mathematics and Computers in Simulation* 55(1-3), 271-280. https://doi.org/10.1016/S0378-4754(00)00270-6

All of the platform's credits and licences are listed on the page **Credits & licences** of the platform.